# 01 — Tải dữ liệu, kiểm kê Shards và xác thực Schema Contract

**Mục tiêu:** Kiểm kê tất cả file nguồn raw CSV, kiểm tra tính toàn vẹn (checksum, size, dòng), đối chiếu schema và chuyển đổi sang Parquet staging.

Single Source of Truth: `PUBG_RESEARCH_SPEC.md` v3.0 | `PUBG_IMPLEMENTATION_PLAN.md`


Chọn `runtime` để chạy không cần Drive, hoặc `drive` để 13 notebook dùng chung dữ liệu bền vững. Với `drive`, mọi notebook phải dùng cùng `PUBG_DRIVE_PROJECT_ROOT` và chạy theo thứ tự.


In [ ]:
# @title Chọn nơi lưu dữ liệu { display-mode: "form" }
# @markdown `runtime`: không cần Drive, phù hợp notebook All-in-One.
# @markdown `drive`: lưu nối tiếp 13 notebook trong cùng thư mục Google Drive.
PUBG_STORAGE_MODE = "runtime"  # @param ["runtime", "drive"]
PUBG_DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/Project_PUBG"  # @param {type:"string"}


In [ ]:
# Bootstrap: runtime mode needs no Drive; drive mode persists stage outputs.
import base64
import importlib.util
import io
import os
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
PUBG_STORAGE_MODE = globals().get("PUBG_STORAGE_MODE", "runtime").strip().lower()
if PUBG_STORAGE_MODE not in {"runtime", "drive"}:
    raise ValueError("PUBG_STORAGE_MODE must be 'runtime' or 'drive'")

if PUBG_STORAGE_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Drive mode is available only on Google Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(globals().get(
        "PUBG_DRIVE_PROJECT_ROOT", "/content/drive/MyDrive/Project_PUBG"
    )).expanduser().resolve()
else:
    _candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/Project_PUBG")]
    _candidates += [p / "Project_PUBG" for p in list(_candidates)]
    PROJECT_ROOT = next((p.resolve() for p in _candidates
                         if (p / "configs/data.yaml").is_file() and (p / "src/utils/config.py").is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / "Project_PUBG"

if not (PROJECT_ROOT / "configs/data.yaml").is_file() or not (PROJECT_ROOT / "src/utils/config.py").is_file():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    _bundle = zipfile.ZipFile(io.BytesIO(base64.b64decode('...')))
    for _entry in _bundle.infolist():
        _target = (PROJECT_ROOT / _entry.filename).resolve()
        if not _target.is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Invalid bundled path")
        if not _target.exists():
            _target.parent.mkdir(parents=True, exist_ok=True)
            _target.write_bytes(_bundle.read(_entry))
    _bundle.close()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if globals().get("PUBG_INSTALL_DEPENDENCIES", IN_COLAB) and not globals().get("_PUBG_PACKAGES_READY", False):
    _requirements = {
        "numpy": "numpy>=1.24.0", "pandas": "pandas>=2.0.0",
        "pyarrow": "pyarrow>=12.0.0", "duckdb": "duckdb>=0.9.0",
        "scipy": "scipy>=1.10.0", "sklearn": "scikit-learn>=1.3.0",
        "yaml": "pyyaml>=6.0",
    }
    _missing = [spec for module, spec in _requirements.items() if importlib.util.find_spec(module) is None]
    if _missing:
        print("Installing missing packages:", ", ".join(_missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--prefer-binary", *_missing])
    _PUBG_PACKAGES_READY = True

from src.utils.config import load_config, resolve_paths
cfg = load_config(str(PROJECT_ROOT / "configs"))
if PUBG_STORAGE_MODE == "drive":
    cfg["paths"]["environments"]["drive"] = {
        "raw_root": str(PROJECT_ROOT / "data/raw"),
        "data_root": str(PROJECT_ROOT / "data"),
        "artifacts_root": str(PROJECT_ROOT / "artifacts"),
        "figures_root": str(PROJECT_ROOT / "figures"),
        "reports_root": str(PROJECT_ROOT / "reports"),
        "temp_dir": globals().get("PUBG_RUNTIME_TEMP_DIR", "/content/temp"),
    }
    cfg["paths"]["active_environment"] = "drive"
paths = resolve_paths(cfg)
for _path in paths.values():
    _path.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_ROOT)
print("Storage:", paths["data_root"], "| Results:", paths["reports_root"])
if PUBG_STORAGE_MODE == "drive":
    print("Storage mode: Google Drive. Stage outputs persist for the next notebook.")
else:
    print("Storage mode: runtime. No Drive authorization required; export before reset.")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()  # bootstrap has located the project and set cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, resolve_paths
from src.data.io import get_duckdb_connection, atomic_write_json
from src.data.inventory import inventory_sources
from src.data.schema import convert_shard_to_parquet, validate_shard_schema
from src.data.checkpoints import CheckpointManager

cfg = load_config(str(PROJECT_ROOT / "configs"))
paths = resolve_paths(cfg)
con = get_duckdb_connection(temp_dir=paths["temp_dir"], **{k: cfg["runtime"]["duckdb"][k] for k in ("memory_limit", "threads")})
ckpt_mgr = CheckpointManager(manifest_path=paths["checkpoints"] / "checkpoint_manifest.json")

In [ ]:
# 1. Khám phá và kiểm kê toàn bộ file nguồn raw
raw_root = Path(paths["raw_root"]).resolve()
data_cfg = cfg["data"]
from src.data.download_data import download_and_extract_archive

inventory = inventory_sources(
    raw_root=raw_root,
    agg_patterns=data_cfg["discovery"]["agg_patterns"],
    kill_patterns=data_cfg["discovery"]["kill_patterns"],
    con=con,
    compute_hash=False
)

# Tự động tải và giải nén nếu chưa có file CSV thô
if inventory["total_files"] == 0 and data_cfg["source"].get("archive_url"):
    print(f"Chưa có dữ liệu thô tại {raw_root}. Đang tải/giải nén từ {data_cfg['source']['archive_url']}...")
    download_and_extract_archive(
        archive_url=data_cfg["source"]["archive_url"],
        target_dir=raw_root,
        expected_checksum=data_cfg["source"].get("archive_sha256"),
        archive_filename=data_cfg["source"].get("archive_filename", "Data_PUBG.zip"),
    )
    inventory = inventory_sources(
        raw_root=raw_root,
        agg_patterns=data_cfg["discovery"]["agg_patterns"],
        kill_patterns=data_cfg["discovery"]["kill_patterns"],
        con=con,
        compute_hash=False
    )

if not inventory["aggregate_shards"] or not inventory["death_shards"]:
    raise FileNotFoundError("Thiếu aggregate hoặc deaths CSV. Kiểm tra raw_root và public archive_url trong configs/data.yaml.")
if any(shard["status"] != "valid" for shard in inventory["aggregate_shards"] + inventory["death_shards"]):
    raise ValueError("Có shard không đọc được. Kiểm tra log inventory trước khi chuyển sang Parquet.")
print(f"Tìm thấy {inventory['total_files']} files ({inventory['total_bytes'] / (1024**3):.2f} GB).")
atomic_write_json(paths["manifests"] / "source_inventory.json", inventory)

In [ ]:
# 2. Chuyển đổi typed Parquet staging
staging_dir = paths["interim"] / "staging_shards"
schema_cfg = cfg["schema"]

converted_agg = []
for shard in inventory["aggregate_shards"]:
    csv_file = Path(raw_root) / shard["relative_path"]
    out_pq = staging_dir / f"{csv_file.stem}.parquet"
    col_map = {c: c for c in schema_cfg["aggregate"]["required_columns"]}
    convert_shard_to_parquet(con, csv_file, out_pq, schema_cfg["aggregate"]["required_columns"], col_map)
    converted_agg.append(out_pq)

converted_kill = []
for shard in inventory["death_shards"]:
    csv_file = Path(raw_root) / shard["relative_path"]
    out_pq = staging_dir / f"{csv_file.stem}.parquet"
    col_map = {c: c for c in schema_cfg["deaths"]["required_columns"]}
    convert_shard_to_parquet(con, csv_file, out_pq, schema_cfg["deaths"]["required_columns"], col_map)
    converted_kill.append(out_pq)

ckpt_mgr.commit("schema", "schema_v1", {"converted_agg_shards": staging_dir})
print("Gate G1 Hoàn tất: Shards đã được kiểm kê và chuẩn hóa sang Parquet.")